# Notebook 0: Theory and Applications. Approach to the problem

## Theory behing the Mousetrap problem
The core idea is that there is a big box filled with mousetraps where two balls are balanced on it. A ball is thrown at random cuch that when it contacts the balls or mousetrap it makes the last clash down, making the other two rise up in the air and carry out a chain reaction.

## Applications of this chain reaction
This type of chain reaction can be used to model many different systems:
- Nuclear fission
- Epidemology (spread of diseases)
- Population dynamics
- Wildfire propagations (mousetraps are trees and ping-pong balls are burning ember)

## Approach to the problem
A simplified way of modelling this could be that we have an array that is divided into cells which are in state zero, but randomly one of them is flipped to one and there is a probability taht that one state will. ake nighboring cells also turn to one with a probabiulity that is inversely proportional to the distance to the distance to the one state.

In [ ]:
import numpy as np

def simulate_ca(rho, p0, kernel='exp', r0=3.0, L=50, steps=200, rng=None):
    rng = rng or np.random.default_rng()
    grid = (rng.random((L, L)) < rho).astype(int)      # 1 = armed trap present, 0 = empty
    state = np.where(grid == 1, 0, -1)                  # 0=armed, 1=fired, -1=no trap
    ys, xs = np.where(grid == 1)
    seed = rng.integers(len(xs))
    state[ys[seed], xs[seed]] = 1 

    # This block defines a grid where -1 means it is permamently inert
    #, 0 means it is armed, 1 means it fired this generation and -2 means it
    # previously fired

    def p_of_d(d):
        if kernel == 'exp': # This is a type of evolution that evolves exponentially, symbolizinf a characteristic exponential decay
            return p0 * np.exp(-d / r0)
        elif kernel == 'inv': # This describes a second type of evolution where there is a distinct distance cutoff
            return np.minimum(p0 / np.maximum(d, 1), 1.0) * (d <= r0)

    # This block taked a numpy array of distances and converts it into a 
    # triggering probability

    for t in range(steps):
        fired = np.argwhere(state == 1)
        if len(fired) == 0:
            break #If there are no active firers, then the system will not change and the current timestep
            # will be outputted
        armed = np.argwhere(state == 0)
        if len(armed) == 0:
            break # Same concept as before, if all the cells have shot out, then 
            # the system will remain constant and the current timestep will be outputted
        for fy, fx in fired:
            d = np.hypot(armed[:,0]-fy, armed[:,1]-fx) # distance from armed cell to taget cell
            hits = rng.random(len(d)) < p_of_d(d) # just a probability
            for (ay, ax) in armed[hits]:
                state[ay, ax] = 1
        state[state == 1] = np.where(rng.random(np.sum(state==1)) < 1, 2, 1)  # mark spent
        state[state == 2] = -2  # spent, distinct from empty
    return state # This is the code that takes care of the actual evolution of the system.
    # The output of the function is the state after the given number of timesteps

print(simulate_ca(0.1, 0.1))

[[-1 -1 -1 ... -1 -1 -1]
 [-1 -1 -1 ... -1 -1 -1]
 [-1 -1 -1 ... -1 -1  0]
 ...
 [-1 -1 -1 ...  0 -1 -1]
 [-1 -1 -1 ... -1 -1 -1]
 [-1  0  0 ... -1 -1 -1]]
